# Chosen Model Analysis

This notebook requires completed, current artifacts from `Models_Pipeline.ipynb`. It does not train or reselect a model. `HYBRID_LGBM40_DEEPSET40_DYNAMIC` is fixed as the hypothesis-driven final specification because it combines the selected expanded tree benchmark with the dynamic market-cloud representation. The analysis compares it with its aligned internal components, a 50/50 blend, and the static hybrid.

Each hybrid refit uses validation years 1--3 for component early stopping and validation year 4 for the frozen convex weight. Regimes use trailing 12-month market volatility and a one-month-lagged expanding historical median threshold. Component importance is reported separately for LightGBM and dynamic DeepSets. Factor attribution uses FF5 plus momentum with Newey--West-6 inference. Economic robustness includes long/short attribution, transaction costs, universe and weighting sensitivity, missing-return stress, and observed-return outlier checks.

In [ ]:
from pathlib import Path
import os
import sys
import importlib

COLAB_PROJECT_DIR = Path('/content/drive/MyDrive/Colab Notebooks/FDS Project')
WINDOWS_PROJECT_DIR = Path('C:/Users/sandh/OneDrive/Documents/Coding/FDS Project')
if Path('/content').exists() and not COLAB_PROJECT_DIR.exists():
    from google.colab import drive
    drive.mount('/content/drive')
PROJECT_DIR = COLAB_PROJECT_DIR if COLAB_PROJECT_DIR.exists() else WINDOWS_PROJECT_DIR
os.chdir(PROJECT_DIR)
project_path = str(PROJECT_DIR)
sys.path = [path for path in sys.path if path != project_path]
sys.path.insert(0, project_path)
for module_name in tuple(sys.modules):
    if module_name == 'src' or module_name.startswith('src.'):
        del sys.modules[module_name]
importlib.invalidate_caches()
print('Project directory:', PROJECT_DIR)

In [ ]:
from src.config import ExperimentConfig

CHOSEN_MODEL_ID = 'HYBRID_LGBM40_DEEPSET40_DYNAMIC'
DATA_PATH = PROJECT_DIR / 'jkp_USA_100chars_1980_2024.parquet'
OUTPUT_DIR = PROJECT_DIR / 'model_runs'
CONFIG = ExperimentConfig(
    experiment_id='core20_benchmarks_v1',
    project_dir=PROJECT_DIR, data_path=DATA_PATH, output_dir=OUTPUT_DIR,
    selected_models=(CHOSEN_MODEL_ID,), seed=42, use_gpu=False,
)
CONFIG.validate()

# Load or refresh cached implementability diagnostics.
from src.portfolio_robustness import run_portfolio_robustness
chosen_robustness = run_portfolio_robustness(
    CONFIG.run_dir, model_ids=(CHOSEN_MODEL_ID,)
)

from src.chosen_model_analysis import validate_chosen_model_artifacts
chosen_artifacts = validate_chosen_model_artifacts(CONFIG, CHOSEN_MODEL_ID)

# Validate the fixed chosen-model entry against the completed comparison.
import pandas as pd
comparison = pd.read_csv(CONFIG.run_dir / 'model_comparison.csv')
chosen_row = comparison.loc[comparison['model_id'].eq(CHOSEN_MODEL_ID)]
if len(chosen_row) != 1:
    raise RuntimeError(f'Expected exactly one comparison row for {CHOSEN_MODEL_ID}.')
best_sharpe_model = comparison.loc[comparison['sharpe'].idxmax(), 'model_id']
print('Chosen model:', CHOSEN_MODEL_ID)
print('Model signature:', chosen_artifacts['model_signature'])
print('Selection rule: pre-specified LGBM40 plus dynamic market-cloud hybrid')
print('Highest realized Sharpe model:', best_sharpe_model)
print('Run directory:', CONFIG.run_dir)
display(chosen_row[[
    'model_id', 'pooled_oos_r2', 'robust_oos_r2', 'mean_monthly_rank_ic',
    'annualized_return', 'annualized_volatility', 'sharpe',
    'newey_west_t_stat', 'max_drawdown', 'hit_rate', 'n_months',
]])

## 1. Signal-date market regimes

In [ ]:
from src.chosen_model_analysis import build_signal_date_regimes, regime_stability

regimes = build_signal_date_regimes(CONFIG)
regime_results = regime_stability(CONFIG, CHOSEN_MODEL_ID, regimes)
display(regime_results)

In [ ]:
import matplotlib.pyplot as plt
regime_results.set_index('regime')[['annualized_return', 'sharpe']].plot.bar(
    subplots=True, layout=(1, 2), figsize=(10, 4), legend=False,
    title=['Annualized return', 'Sharpe ratio']
)
plt.suptitle(f'{CHOSEN_MODEL_ID}: stability across signal-date volatility regimes')
plt.tight_layout(); plt.show()

## 2. Incremental hybrid value

The weighted hybrid is compared with its identically trained LightGBM and dynamic DeepSets components and with a simple 50/50 blend. Component fitting uses validation years 1--3, while validation year 4 is reserved exclusively for estimating the hybrid weight. Paired inference uses monthly Newey--West tests, test-year-clustered rank-IC tests, year-block bootstrap Sharpe intervals, and Holm-adjusted p-values.

In [ ]:
from src.chosen_model_analysis import aligned_hybrid_component_analysis

aligned_tests, annual_consistency, aligned_monthly = aligned_hybrid_component_analysis(
    CONFIG, CHOSEN_MODEL_ID, regimes=regimes, bootstrap_draws=2000
)
display(aligned_tests)
display(annual_consistency)

## 3. Validation-weight stability

The component checkpoints are selected on validation years 1--3. The convex LGBM and DeepSets weights are then estimated only on validation year 4 and frozen for the following test year.

In [ ]:
from src.chosen_model_analysis import hybrid_validation_weights

hybrid_weights = hybrid_validation_weights(CONFIG, CHOSEN_MODEL_ID)
display(hybrid_weights)
display(hybrid_weights[['weight_lgbm', 'weight_deepset', 'weight_observations']].describe())
hybrid_weights.set_index('test_year')[['weight_lgbm', 'weight_deepset']].plot(
    figsize=(10, 4), marker='o', title='Validation-selected hybrid weights by test year'
)
plt.ylabel('Convex weight'); plt.ylim(-0.02, 1.02); plt.tight_layout(); plt.show()

## 4. Hybrid component importance by regime

LightGBM importance is measured from absolute tree contributions within each regime. Dynamic DeepSets importance groups each characteristic's current, lagged, and velocity coordinates and permutes that group across firms within each month. The two importance scales are reported separately and are not presented as a single ensemble-wide ranking.

In [ ]:
from src.chosen_model_analysis import hybrid_component_feature_importance_by_regime

importance = hybrid_component_feature_importance_by_regime(CONFIG, CHOSEN_MODEL_ID, regimes)
top_importance = importance.query('importance_rank <= 10')
display(top_importance)

In [ ]:
for component_id, component in top_importance.groupby('component_model_id'):
    pivot = component.pivot(index='feature', columns='regime', values='importance_share').fillna(0)
    pivot.plot.barh(figsize=(9, 7), title=f'{component_id}: importance by regime')
    plt.xlabel('Within-component importance share'); plt.tight_layout(); plt.show()

## 5. FF5 plus momentum factor decomposition

Monthly US FF5 and momentum factors are downloaded from Kenneth French's data library for ex-post return attribution. These data do not enter model fitting, validation, or portfolio formation.

In [ ]:
from src.chosen_model_analysis import download_ff5_momentum, factor_decomposition

factors = download_ff5_momentum()
factor_results = factor_decomposition(CONFIG, CHOSEN_MODEL_ID, factors)
display(factor_results)

## 6. Long-versus-short attribution and final cost discussion

In [ ]:
from IPython.display import Markdown, display
from src.chosen_model_analysis import long_short_and_cost_attribution

attribution, cost_discussion = long_short_and_cost_attribution(CONFIG, CHOSEN_MODEL_ID)
display(attribution)
display(Markdown(cost_discussion))

## 7. Final audit, frozen manifest, and report-ready outputs

This step validates model coverage, prediction schemas, signatures, diagnostics, target timing, portfolio reconstruction, robustness outputs, and declared comparisons. It then writes the frozen research manifest and report-ready tables and figures without modifying model predictions.

In [ ]:
from src.project_finalization import (
    run_final_project_audit, write_frozen_manifest, generate_report_outputs,
)

FINAL_MODEL_IDS = (
    'LASSO_20', 'LGBM_20', 'XGBOOST_20', 'NN2_20', 'NN2_40', 'NN3_20', 'NN4_20',
    'LGBM_40', 'LGBM_60', 'LGBM_80', 'LGBM_100',
    'LGBM_20_LAG1', 'LGBM_20_LAG2',
    'LGBM_40_LAG1', 'LGBM_40_LAG2',
    'MLP_40', 'DEEPSET_40',
    'DEEPSET_40_LAG1', 'DEEPSET_40_DYNAMIC',
    'HYBRID_MLP40_DEEPSET40',
    'HYBRID_LGBM40_DEEPSET40',
    'HYBRID_LGBM40_DEEPSET40_DYNAMIC',
)
final_audit = run_final_project_audit(CONFIG, FINAL_MODEL_IDS, CHOSEN_MODEL_ID)
display(final_audit.loc[~final_audit['passed']])
if not final_audit['passed'].all():
    raise RuntimeError('Final project audit has failed checks; review the table above.')
frozen_manifest = write_frozen_manifest(CONFIG, FINAL_MODEL_IDS, CHOSEN_MODEL_ID)
report_files = generate_report_outputs(CONFIG, CHOSEN_MODEL_ID)
print('Final audit passed:', len(final_audit), 'checks')
print('Frozen chosen model:', frozen_manifest['chosen_model_id'])
display(pd.DataFrame({'report_file': list(report_files)}))

## Interpretation checklist

- Does performance and rank IC remain positive in both volatility regimes?
- Do the leading grouped permutation-importance characteristics remain economically interpretable across regimes?
- Is FF5+momentum alpha positive with a reliable HAC t-stat?
- Is performance balanced across the long and short legs?
- Does the strategy remain attractive under the 25 bps scenario, outside microcaps, and under the adverse missing-return stress?
- State explicitly that model selection preceded this analysis; these results explain and stress-test the chosen model rather than reopen the model search.